# Bank Marketing — Model Experiments

This notebook trains all classifiers via the pipeline and inspects results interactively.
MLflow tracks every run automatically.

**MLflow UI:** http://localhost:5000 (requires `make infra-up`)

Run from the repo root:
```bash
uv run jupyter lab experiments/notebooks/02_model_experiments.ipynb
```

In [ ]:
import sys
sys.path.insert(0, '../..')

import warnings
warnings.filterwarnings('ignore')

import mlflow
import pandas as pd
import matplotlib.pyplot as plt

from src.config import settings
from src.data.preprocess import load_processed
from src.models.train import train_all_models
from src.models.evaluate import generate_evaluation_report
from src.visualization.plots import plot_feature_importance, plot_model_comparison

print(f'MLflow tracking URI: {settings.mlflow_tracking_uri}')

## 1 — Load Processed Data

Make sure you have run `make preprocess` (or `make pipeline`) first.

In [ ]:
train_df, val_df, test_df = load_processed()
print(f'Train: {train_df.shape}  |  Val: {val_df.shape}  |  Test: {test_df.shape}')
print(f'Positive rate — Train: {train_df["subscribed"].mean():.2%}, '
      f'Val: {val_df["subscribed"].mean():.2%}, Test: {test_df["subscribed"].mean():.2%}')

## 2 — Train All Models

This trains Logistic Regression, Random Forest, and XGBoost,
logging all runs to MLflow.

In [ ]:
results = train_all_models(train_df, val_df)

## 3 — Validation Metric Summary

In [ ]:
summary = pd.DataFrame([
    {'model': r['name'], 'run_id': r['run_id'], **r['metrics']}
    for r in results
])
summary

In [ ]:
fig = plot_model_comparison(summary)
plt.show()

## 4 — Feature Importance (Best Model)

In [ ]:
best = max(results, key=lambda r: r['metrics']['roc_auc'])
print(f'Best model: {best["name"]}  |  Val ROC-AUC: {best["metrics"]["roc_auc"]:.4f}')
fig = plot_feature_importance(best['pipeline'], best['name'])
plt.show()

## 5 — Test Set Evaluation (Best Model)

In [ ]:
report = generate_evaluation_report(best['pipeline'], test_df, best['name'])
print(report['report_text'])

In [ ]:
# Display saved ROC curve
from IPython.display import Image
Image(report['roc_path'])

In [ ]:
Image(report['cm_path'])

## 6 — Threshold Analysis

The default 0.5 threshold may not be optimal for this imbalanced dataset.
Review the threshold analysis table to pick the best operating point for your business goal.

- **Maximise recall** → lower threshold (catch more subscribers, accept more false positives)
- **Maximise precision** → higher threshold (fewer false positives, may miss some subscribers)

In [ ]:
threshold_df = report['threshold_analysis']
ax = threshold_df.set_index('threshold')[['precision', 'recall', 'f1']].plot(
    figsize=(9, 4), title='Precision / Recall / F1 vs Decision Threshold'
)
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
plt.tight_layout()
plt.show()